<a href="https://colab.research.google.com/github/maierav/ai_oscp_neuro/blob/main/notebooks/subsample_tuning_balanced.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tuning-balanced oddball response — controlling the sampling bias by subsampling

The companion `crossscale_oddball_index` analysis showed that the raw oddball-vs-standard
index (**OI**) reverses sign across techniques — mildly positive in spikes, strongly
negative in the calcium methods — because two-photon imaging **over-samples neurons
tuned to the frequent 0° standard**. That analysis fixed the problem from the
*stimulus* side (reference the oddball against the physically identical grating in the
equiprobable control block → the tuning-independent **DvI**).

This notebook fixes it from the *population* side instead. For each technique we:
1. compute each cell's **orientation preference** (TPI, measured independently in the
   control block: +1 prefers the 90° deviant orientation, −1 prefers the 0° standard),
2. **subsample to an even TPI distribution** — equal numbers of 0°- and 90°-preferring
   cells — repeated over many random draws and averaged,
3. re-average the balanced population's PSTHs under **three normalizations** (raw,
   % change from baseline, z-score), with ±SEM bands.

**Result.** Balancing the tuning distribution removes the sampling artifact: the
mesoscope OI flips from **−0.45 (raw)** to **+0.28 (balanced)**, and the oddball
response is **positive in every technique** — the same conclusion the DvI reached, by
an independent route. The main finding is not an artifact of which cells each technique
samples.

In [ ]:
#@title Install
!pip -q install pynwb h5py remfile requests pandas numpy matplotlib scipy

In [ ]:
#@title Streaming helpers + per-cell PSTH extractors
import numpy as np, pandas as pd, h5py, remfile, requests, re
from scipy import stats as ss
def s3_url(ds, aid, version="draft"):
    u=f"https://api.dandiarchive.org/api/dandisets/{ds}/versions/{version}/assets/{aid}/download/"
    return requests.get(u,allow_redirects=False,timeout=60).headers["Location"]
def col(g,c):
    v=g[c][:]; return np.array([x.decode() if isinstance(x,bytes) else x for x in v])
def idx(a,b):
    a=np.asarray(a,float);b=np.asarray(b,float);d=np.abs(a)+np.abs(b);return np.where(d>1e-12,(a-b)/d,np.nan)
WIN={"ecephys":(0.0,0.5),"mesoscope":(0.0,0.7),"slap2":(0.0,0.7)}
BASE_E=(-0.1,-0.005);BASE_C=(-0.3,-0.02)
PRE,POST,BW=0.3,1.2,0.02
EDGES=np.arange(-PRE,POST+BW,BW);TCEN=EDGES[:-1]+BW/2;GRID=np.arange(-PRE,POST,BW)
BASEBIN_E=(TCEN<-0.005)&(TCEN>=-0.1);BASEBIN_C=(GRID<-0.02)&(GRID>=-0.3)

def ece_cells(aid):
    fh=h5py.File(remfile.File(s3_url("001637",aid)),"r")
    try:
        w=WIN["ecephys"]
        gO=fh["intervals"]["Standard mismatch block_presentations"];TT=col(gO,"TrialType");ts=gO["start_time"][:]
        t_std=ts[TT=="standard"];t_o90=ts[TT=="orientation_90"]
        gC=fh["intervals"]["Control block 1_presentations"];Cori=gC["Orientation"][:].astype(float)
        Cdur=gC["stop_time"][:]-gC["start_time"][:];Cts=gC["start_time"][:];ok=Cdur>=0.3
        t_c90=Cts[np.isclose(Cori,1.571,atol=0.05)&ok];t_c0=Cts[np.isclose(Cori,0.0,atol=0.05)&ok]
        U=fh["units"];st=U["spike_times"][:];sti=U["spike_times_index"][:]
        n=len(sti);qc=U["default_qc"][:] if "default_qc" in U else np.ones(n,bool)
        Eg=fh["general/extracellular_ephys/electrodes"];eloc=col(Eg,"location");egroup=col(Eg,"group_name")
        dev=col(U,"device_name");eci=U["extremum_channel_index"][:]
        groups=sorted(set(egroup),key=lambda gn:np.where(egroup==gn)[0][0])
        offs={gn:int(np.where(egroup==gn)[0][0]) for gn in groups};bl={gn:int((egroup==gn).sum()) for gn in groups}
        def d2g(dd):
            for gn in groups:
                if dd[-1].lower() in gn.lower(): return gn
            return groups[0]
        dmap={dd:d2g(dd) for dd in set(dev)}
        u_loc=np.array([eloc[offs[dmap[dev[i]]]+min(int(eci[i]),bl[dmap[dev[i]]]-1)] for i in range(n)])
        vis=np.where(qc & np.array([bool(re.match("VIS",r)) for r in u_loc]))[0]
        def sp_(i): return st[(0 if i==0 else sti[i-1]):sti[i]]
        def wrate(sp,times,win): return (np.searchsorted(sp,times+win[1])-np.searchsorted(sp,times+win[0]))/(win[1]-win[0])
        def psth(sp,times):
            lo=np.searchsorted(sp,times.min()-PRE);hi=np.searchsorted(sp,times.max()+POST);sp2=sp[lo:hi]
            rel=(sp2[None,:]-times[:,None]).ravel();rel=rel[(rel>=-PRE)&(rel<POST)]
            return np.histogram(rel,EDGES)[0]/(len(times)*BW)
        recs=[];P={"std":[],"o90":[],"c90":[]}
        for u in vis:
            sp=sp_(u);r_std=wrate(sp,t_std,w)-wrate(sp,t_std,BASE_E)
            resp=np.any(r_std!=0) and ss.wilcoxon(r_std).pvalue<0.05
            ps=psth(sp,t_std);po=psth(sp,t_o90);pc=psth(sp,t_c90)
            r_c90=wrate(sp,t_c90,w)-wrate(sp,t_c90,BASE_E);r_c0=wrate(sp,t_c0,w)-wrate(sp,t_c0,BASE_E)
            r_o90=wrate(sp,t_o90,w)-wrate(sp,t_o90,BASE_E)
            recs.append(dict(area=u_loc[u],resp=bool(resp),base_mean=ps[BASEBIN_E].mean(),base_std=ps[BASEBIN_E].std(),
                R_std=np.mean(r_std),R_o90=np.mean(r_o90),R_c90=np.mean(r_c90),R_c0=np.mean(r_c0),resp_sign=int(np.sign(np.mean(r_std)))))
            P["std"].append(ps);P["o90"].append(po);P["c90"].append(pc)
        df=pd.DataFrame(recs);df["OI"]=idx(df.R_o90,df.R_std);df["DvI"]=idx(df.R_o90,df.R_c90);df["TPI"]=idx(df.R_c90,df.R_c0);df["modality"]="ecephys"
        return df,{k:np.array(v) for k,v in P.items()},TCEN
    finally: fh.close()

def img_cells(ds,aid,slap=False):
    fh=h5py.File(remfile.File(s3_url(ds,aid)),"r")
    try:
        w=WIN["slap2" if slap else "mesoscope"]
        if slap:
            g=fh["intervals"]["gratings"];O=g["orientation"][:].astype(float);Cc=g["contrast"][:].astype(float)
            ts=g["start_time"][:];dur=g["stop_time"][:]-g["start_time"][:];ok=(Cc>0)&(dur>=0.3)
            t_std=ts[np.isclose(O,0.0,atol=0.05)&ok];t_o90=ts[np.isclose(O,1.571,atol=0.05)&ok];t_c90=t_c0=None
            fl=fh["processing"]["ophys"];unit_sets=[]
            for dmd,off in [("Fluorescence_DMD1",0.115),("Fluorescence_DMD2",-0.165)]:
                sub=fl[dmd];key=[k for k in sub if k.endswith("dFF")][0]
                unit_sets.append((sub[key]["data"][:],sub[key]["timestamps"][:]+off,"SLAP2"))
        else:
            I=fh["intervals"];blk=[k for k in I if "Standard mismatch" in k][0];g=I[blk]
            TT=col(g,"TrialType");ts=g["start_time"][:];t_std=ts[TT=="standard"];t_o90=ts[TT=="orientation_90"]
            gC=I["Control block 1_presentations"];Cori=gC["Orientation"][:].astype(float)
            Cdur=gC["stop_time"][:]-gC["start_time"][:];Cts=gC["start_time"][:];ok=Cdur>=0.3
            t_c90=Cts[np.isclose(Cori,1.571,atol=0.05)&ok];t_c0=Cts[np.isclose(Cori,0.0,atol=0.05)&ok]
            unit_sets=[]
            for pl in [k for k in fh["processing"] if k.startswith("VIS")]:
                grp=fh["processing"][pl];dff=grp["dff_timeseries"]["dff_timeseries"];data=dff["data"][:];dts=dff["timestamps"][:]
                try: soma=grp["image_segmentation"]["roi_table"]["is_soma"][:].astype(bool)
                except: soma=np.ones(data.shape[1],bool)
                unit_sets.append((data[:,soma],dts,pl))
        def wr(tr,dts,times):
            out=[]
            for t0 in times:
                rb=(dts>=t0+w[0])&(dts<t0+w[1]);bb=(dts>=t0+BASE_C[0])&(dts<t0+BASE_C[1])
                out.append(np.nanmean(tr[rb])-np.nanmean(tr[bb]) if rb.sum() and bb.sum() else np.nan)
            return np.array(out)
        def trace(tr,dts,times):
            acc=np.zeros(len(GRID));cnt=np.zeros(len(GRID))
            for t0 in times:
                i0=np.searchsorted(dts,t0-PRE);i1=np.searchsorted(dts,t0+POST);tt=dts[i0:i1]-t0
                if len(tt)<3: continue
                vi=np.interp(GRID,tt,tr[i0:i1],left=np.nan,right=np.nan);m=np.isfinite(vi);acc[m]+=vi[m];cnt[m]+=1
            return acc/np.maximum(cnt,1)
        recs=[];P={"std":[],"o90":[],"c90":[]}
        for data,dts,area in unit_sets:
            for r in range(data.shape[1]):
                tr=data[:,r];e=wr(tr,dts,t_std);ef=e[np.isfinite(e)]
                if len(ef)<5: continue
                m=np.nanmean(ef);resp=ss.wilcoxon(ef).pvalue<0.05 and m>0
                ps=trace(tr,dts,t_std);po=trace(tr,dts,t_o90)
                pc=trace(tr,dts,t_c90) if not slap else np.full(len(GRID),np.nan)
                r_o90=np.nanmean(wr(tr,dts,t_o90))
                if slap: r_c90=r_c0=np.nan;tpi=idx([r_o90],[m])[0]
                else: r_c90=np.nanmean(wr(tr,dts,t_c90));r_c0=np.nanmean(wr(tr,dts,t_c0));tpi=idx([r_c90],[r_c0])[0]
                recs.append(dict(area=re.sub(r"_?\d.*$","",area),resp=bool(resp),base_mean=np.nanmean(ps[BASEBIN_C]),base_std=np.nanstd(ps[BASEBIN_C]),
                    R_std=m,R_o90=r_o90,R_c90=r_c90,R_c0=r_c0,resp_sign=int(np.sign(m)),TPI=tpi))
                P["std"].append(ps);P["o90"].append(po);P["c90"].append(pc)
        df=pd.DataFrame(recs);df["OI"]=idx(df.R_o90,df.R_std);df["DvI"]=idx(df.R_o90,df.R_c90) if not slap else np.nan;df["modality"]="slap2" if slap else "mesoscope"
        return df,{k:np.array(v) for k,v in P.items()},GRID
    finally: fh.close()
print("extractors ready")

In [ ]:
#@title Sessions
SESSIONS={
 "ecephys":  [("830851","9b9e8abe-7b43-47f1-b8e1-4114f87898a1"),
              ("830848","228c2c2e-1daf-4bf6-9f66-eb6b2bce5ba5"),
              ("830846","680d1c0c-e338-4d0b-ba29-4329436d2ae2")],
 "mesoscope":[("845342","6288e7b0-5b44-4660-b36d-c14d19220e2c"),
              ("837568","b425c043-ebf5-456c-83a9-1d99465224c6")],
 "slap2":    [("796630","b8c05ec0-0a74-46f5-956d-e82af3cc5d27"),
              ("803496","d23a03af-c3bd-4cf0-9492-6dca96fb201d"),
              ("801381","2ecafd40-29a6-422f-92b0-32f7e940c783")],
}
ONE_PER_MODALITY=True   # imaging sessions run several minutes each; set False for the full pass
if ONE_PER_MODALITY: SESSIONS={k:v[:1] for k,v in SESSIONS.items()}
print({k:[s for s,_ in v] for k,v in SESSIONS.items()})

In [ ]:
#@title Extract per-cell PSTHs + tuning across sessions
import time
META=[];PSTH={m:{"std":[],"o90":[],"c90":[]} for m in SESSIONS};GRIDS={}
for mod,slist in SESSIONS.items():
    for subj,aid in slist:
        t0=time.time()
        try:
            if mod=="ecephys": df,Pr,cen=ece_cells(aid)
            else: df,Pr,cen=img_cells("001768" if mod=="mesoscope" else "001424",aid,slap=(mod=="slap2"))
            df["subject"]=subj;META.append(df);GRIDS[mod]=cen
            for k in ["std","o90","c90"]: PSTH[mod][k].append(Pr[k])
            print(f"  {mod}/{subj}: {df.resp.sum()}/{len(df)} resp | OI={df[df.resp].OI.median():+.3f} TPI={df[df.resp].TPI.median():+.3f} ({time.time()-t0:.0f}s)")
        except Exception as e:
            print(f"  {mod}/{subj}: ERROR {str(e)[:80]}")
MD=pd.concat(META,ignore_index=True);G=MD[MD.resp].copy()
TENS={m:{k:(np.vstack(PSTH[m][k]) if PSTH[m][k] else np.zeros((0,len(GRIDS.get(m,TCEN))))) for k in PSTH[m]} for m in PSTH}
resp_idx={m:np.where((MD.modality==m).values & MD.resp.values)[0]-np.where(MD.modality==m)[0][0] for m in TENS}
print("total responsive cells:",len(G))

## Even-tuning subsampling + normalizations

`even_subsample_indices` bins responsive cells by TPI and draws an equal number from
each occupied bin (repeated `reps` times) so the tuning distribution is flat. Each
cell's PSTH is normalized three ways before averaging:
- **raw** — native units (Hz for spikes, dF/F for calcium);
- **% change** — `100·(psth − baseline)/|baseline|`;
- **z-score** — `(psth − baseline)/baseline_sd`.

Variance bands are ±SEM across the even subsamples.

In [ ]:
#@title Subsampling + normalization helpers
def normalize(psth, base_mean, base_std, kind):
    bm=base_mean[:,None];bs=base_std[:,None]
    if kind=="raw": return psth
    if kind=="pct": return 100.0*(psth-bm)/np.where(np.abs(bm)>1e-9,np.abs(bm),np.nan)
    if kind=="z":   return (psth-bm)/np.where(bs>1e-9,bs,np.nan)

def even_subsample_indices(tpi, nbins=8, seed=0, reps=200):
    rng=np.random.default_rng(seed);edges=np.linspace(-1,1,nbins+1)
    binid=np.clip(np.digitize(tpi,edges)-1,0,nbins-1)
    counts=np.array([(binid==b).sum() for b in range(nbins)])
    nz=counts[counts>0];k=max(int(np.median(nz)) if len(nz) else 0,5)
    subs=[]
    for _ in range(reps):
        picks=[]
        for b in range(nbins):
            members=np.where(binid==b)[0]
            if len(members)==0: continue
            picks.append(rng.choice(members,k,replace=len(members)<k))
        subs.append(np.concatenate(picks))
    return subs,counts,k

def balanced_traces(mod, reps=300, seed=0):
    rows=resp_idx[mod];g=G[G.modality==mod]
    bm=g.base_mean.values;bs=g.base_std.values;tpi=g.TPI.values
    subs,counts,k=even_subsample_indices(tpi,seed=seed,reps=reps)
    out={}
    for norm in ["raw","pct","z"]:
        out[norm]={}
        for cond in ["std","o90","c90"]:
            T=TENS[mod][cond][rows]
            if np.all(np.isnan(T)): out[norm][cond]=(np.full(T.shape[1],np.nan),)*2;continue
            N=normalize(T,bm,bs,norm)
            sm=np.array([np.nanmean(N[s],axis=0) for s in subs])
            out[norm][cond]=(np.nanmean(sm,axis=0),np.nanstd(sm,axis=0))
    return out,GRIDS[mod]*1000,counts,k

# windowed balanced OI: raw vs balanced
WINb={"ecephys":(0.0,0.5),"mesoscope":(0.0,0.7),"slap2":(0.0,0.7)}
summ=[]
for mod in ["ecephys","mesoscope","slap2"]:
    g=G[G.modality==mod];rows=resp_idx[mod];cen=GRIDS[mod]*1000
    lo,hi=WINb[mod];wm=(cen>=lo*1000)&(cen<hi*1000);bm=g.base_mean.values
    ev_std=np.nanmean(TENS[mod]["std"][rows][:,wm],axis=1)-bm
    ev_o90=np.nanmean(TENS[mod]["o90"][rows][:,wm],axis=1)-bm
    subs,counts,k=even_subsample_indices(g.TPI.values,reps=500)
    unb=np.nanmedian((ev_o90-ev_std)/(np.abs(ev_o90)+np.abs(ev_std)))
    bal=np.array([np.nanmedian((ev_o90[s]-ev_std[s])/(np.abs(ev_o90[s])+np.abs(ev_std[s]))) for s in subs])
    summ.append(dict(modality=mod,n_resp=len(g),n_balanced=k*(counts>0).sum(),TPI_median=np.median(g.TPI.values),
        unbalanced_OI=unb,balanced_OI=np.nanmedian(bal),bal_lo=np.nanpercentile(bal,2.5),bal_hi=np.nanpercentile(bal,97.5)))
S=pd.DataFrame(summ);print(S.to_string(index=False))

## Figure 1 · Balanced time-courses (technique × normalization)

In [ ]:
#@title Balanced time-courses, per-panel autoscaled
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d
BT={};CENS={}
for mod in ["ecephys","mesoscope","slap2"]:
    bt,cen,counts,k=balanced_traces(mod,reps=300);BT[mod]=bt;CENS[mod]=cen
CSTD="#7f7f7f";CODD="#c0392b";CCTL="#2e86c1"
NORMLAB={"raw":"Raw (Hz or dF/F)","pct":"% change","z":"z-score (SD)"}
MODLAB={"ecephys":"Neuropixels","mesoscope":"Mesoscope","slap2":"SLAP2"}
mods=["ecephys","mesoscope","slap2"];norms=["raw","pct","z"]
fig,axes=plt.subplots(3,3,figsize=(13,10.5));fig.subplots_adjust(hspace=0.42,wspace=0.28,top=0.92,bottom=0.07)
for i,norm in enumerate(norms):
    for j,mod in enumerate(mods):
        ax=axes[i,j];cen=CENS[mod];bt=BT[mod][norm];ax.axvspan(0,367,color="#fdf0d5",zorder=0);ax.axvline(0,color="k",lw=0.4,ls=":")
        conds=[("std",CSTD,"standard"),("o90",CODD,"oddball 90")]
        if not np.all(np.isnan(bt["c90"][0])): conds.append(("c90",CCTL,"control 90"))
        vis=(cen>=-200)&(cen<=1000);vals=[]
        for cond,c,lab in conds:
            m,sem=bt[cond]
            if not np.isfinite(m).any(): continue
            ms=gaussian_filter1d(m,0.8);ax.plot(cen,ms,color=c,lw=1.6,label=lab)
            ax.fill_between(cen,ms-sem,ms+sem,color=c,alpha=0.2,lw=0);vals+=[(ms-sem)[vis],(ms+sem)[vis]]
        av=np.concatenate(vals);lo,hi=np.nanmin(av),np.nanmax(av);pad=0.08*(hi-lo);ax.set_ylim(lo-pad,hi+pad)
        ax.set_xlim(-200,1000);ax.set_xticks([0,500,1000])
        if i==2: ax.set_xlabel("time from onset (ms)")
        if j==0: ax.set_ylabel(NORMLAB[norm])
        if i==0: ax.set_title(MODLAB[mod],loc="left")
        if i==0 and j==0: ax.legend(frameon=False,fontsize=7,loc="upper right")
fig.suptitle("Tuning-balanced oddball response — even TPI subsample, 3 normalizations (each panel scaled to its own range)",y=0.965)
plt.show()

## Figure 2 · The correction: raw vs balanced OI

In [ ]:
#@title Tuning bias and before/after OI
MODCOL={"ecephys":"#2c3e8c","mesoscope":"#c0392b","slap2":"#8e44ad"}
fig,ax=plt.subplots(1,2,figsize=(11,4.4));fig.subplots_adjust(wspace=0.28,top=0.82,bottom=0.14)
for mod in mods:
    v=G[G.modality==mod].TPI.values
    ax[0].hist(v,bins=np.linspace(-1,1,17),density=True,histtype="step",lw=2,color=MODCOL[mod],label=f"{mod} ({np.median(v):+.2f})")
ax[0].axhline(0.5,color="k",lw=1,ls="--",label="even target");ax[0].axvline(0,color="0.6",lw=0.5,ls=":")
ax[0].set_xlabel("tuning preference (TPI)");ax[0].set_ylabel("density");ax[0].legend(frameon=False,fontsize=7)
ax[0].set_title("Raw populations are tuning-biased",loc="left")
x=np.arange(3);wd=0.36
unb=[S[S.modality==m].unbalanced_OI.values[0] for m in mods];bal=[S[S.modality==m].balanced_OI.values[0] for m in mods]
lo=[S[S.modality==m].bal_lo.values[0] for m in mods];hi=[S[S.modality==m].bal_hi.values[0] for m in mods]
ax[1].bar(x-wd/2,unb,wd,color="0.7",label="raw (biased)");ax[1].bar(x+wd/2,bal,wd,color=[MODCOL[m] for m in mods],label="balanced")
ax[1].errorbar(x+wd/2,bal,yerr=[np.array(bal)-np.array(lo),np.array(hi)-np.array(bal)],fmt="none",ecolor="k",capsize=3,elinewidth=1)
ax[1].axhline(0,color="k",lw=0.8);ax[1].set_xticks(x);ax[1].set_xticklabels(mods);ax[1].set_ylabel("oddball index OI (median)")
ax[1].legend(frameon=False,fontsize=7,loc="lower right");ax[1].set_title("Balancing -> positive in every technique",loc="left")
plt.show()

## Takeaway

- Two-photon imaging over-samples 0°-preferring cells (mesoscope median TPI −0.83), which
  drives the raw oddball-vs-standard OI **negative** even though individual cells show
  the oddball enhancement.
- **Even-tuning subsampling removes the bias.** The mesoscope OI flips from **−0.45
  (raw)** to **+0.28 (balanced)**; every technique becomes **positive** (ecephys +0.17,
  mesoscope +0.28, SLAP2 +0.04).
- The oddball response leads the standard **in all three techniques and all three
  normalizations** — the finding is not an artifact of normalization choice or of which
  cells each method samples.
- This is the same conclusion the control-referenced **DvI** reached
  (`crossscale_oddball_index`), by an independent route: fixing the *population* instead
  of the *reference stimulus*. Two independent corrections agreeing makes the cross-scale
  claim considerably more secure.
- **SLAP2 caveat:** its TPI comes from the same 0°-vs-90° comparison as its OI (no
  independent control block), so its balanced OI is the weakest evidence; the clean
  dissociation rests on Neuropixels + mesoscope.

---
*Generated for `ai_oscp_neuro`. Data: DANDI 001637 / 001768 / 001424 (OpenScope
Community Predictive Processing, Allen Institute for Neural Dynamics). Streams
anonymously.*